In [1]:
import sys
print(sys.executable)

/Users/macpro/Documents/MyProject/venv/bin/python


In [1]:
import pandas as pd
from cassandra.cluster import Cluster

In [2]:
print("Connecting to Cassandra cluster...")
# Connect to the local Docker cluster
cluster = Cluster(['127.0.0.1'], port=9042)
session = cluster.connect('adtech')
print("Connected to keyspace 'adtech' successfully!")

Connecting to Cassandra cluster...
Connected to keyspace 'adtech' successfully!


In [3]:
print("Loading CSV files...")

events = pd.read_csv('/Users/macpro/Downloads/events_header.csv')
users = pd.read_csv('/Users/macpro/Downloads/users.csv')
campaigns = pd.read_csv('/Users/macpro/Downloads/campaigns.csv')

print("Data loaded successfully!")
# Display the first few rows to verify
display(events.head(2))

Loading CSV files...
Data loaded successfully!


,EventID,AdvertiserName,CampaignName,CampaignStartDate,CampaignEndDate,CampaignTargetingCriteria,CampaignTargetingInterest,CampaignTargetingCountry,AdSlotSize,UserID,Device,Location,Timestamp,BidAmount,AdCost,WasClicked,ClickTimestamp,AdRevenue,Budget,RemainingBudget
0,e067dae7-9aaa-4b9c-9195-7af63ce89216,Advertiser_30,Campaign_278,2024-10-10,2024-11-12,Age 29-46,Gaming,Germany,300x250,289903,Mobile,USA,2024-10-31T06:56:39,4.6,3.70,False,NaN,0.0,279306.6,279306.6
1,ea95edfa-1ba0-444d-a40d-a3f064dd5791,Advertiser_25,Campaign_235,2024-10-24,2024-11-28,Age 22-33,Gaming,Australia,728x90,145276,Desktop,UK,2024-11-13T09:19:34,3.5,2.85,False,NaN,0.0,88914.6,88914.6


In [9]:
print("Merging and transforming data...")

# 1. Rename 'Location' in Users to 'region'
users_clean = users.rename(columns={'Location': 'region'})

# 2. Merge events with users on 'UserID' (from your latest list)
df = events.merge(users_clean, on='UserID', how='left')

# 3. Handle CampaignID and merge
# Renaming CampaignID to campaign_id if it exists to match our logic
if 'CampaignID' in campaigns.columns:
    campaigns = campaigns.rename(columns={'CampaignID': 'campaign_id'})

# Now merge on 'CampaignName'
df = df.merge(campaigns[['campaign_id', 'CampaignName']], on='CampaignName', how='left')

# 4. Standardize Timestamps (Note capital 'T' in 'Timestamp')
df['event_timestamp'] = pd.to_datetime(df['Timestamp'])
df['event_date'] = df['event_timestamp'].dt.date.astype(str)

# 5. Global constants
TIME_WINDOW = 'last_30_days'

print(f"Data transformation complete! Total rows: {len(df)}")

Merging and transforming data...
Data transformation complete! Total rows: 10000000


In [12]:
print("Preparing CQL queries for fast insertion...")

insert_q1 = session.prepare("INSERT INTO campaign_daily_performance (event_date, campaign_name, total_impressions, total_clicks, ctr) VALUES (?, ?, ?, ?, ?)")
insert_q2 = session.prepare("INSERT INTO top_advertisers_by_spend (time_window, total_spend, advertiser_name) VALUES (?, ?, ?)")
insert_q3 = session.prepare("INSERT INTO user_ad_history (user_id, event_timestamp, campaign_name, was_clicked) VALUES (?, ?, ?, ?)")
insert_q4 = session.prepare("INSERT INTO top_users_by_clicks (time_window, total_clicks, user_id) VALUES (?, ?, ?)")
insert_q5 = session.prepare("INSERT INTO top_advertisers_by_region (region, time_window, total_spend, advertiser_name) VALUES (?, ?, ?, ?)")

print("Queries prepared!")

Preparing CQL queries for fast insertion...
Queries prepared!


In [16]:
print("Executing Task 1: CTR per campaign per day...")


df['click_num'] = df['WasClicked'].apply(lambda x: 1 if x in [1, '1', True, 'True', 'true'] else 0)


task1_df = df.groupby(['event_date', 'campaign_id']).agg(
    impression=('EventID', 'count'),
    click=('click_num', 'sum')
).reset_index()

# count CTR
for _, row in task1_df.iterrows():
    ctr = (row['click'] / row['impression']) if row['impression'] > 0 else 0.0
    session.execute(insert_q1, (
        row['event_date'],
        str(row['campaign_id']),
        int(row['impression']),
        int(row['click']),
        float(ctr)
    ))

print("Task 1 completed!")

Executing Task 1: CTR per campaign per day...
Task 1 completed!


In [17]:
#check the results for 1st task
query = "SELECT * FROM campaign_daily_performance LIMIT 10;"

print("data insight CassandraDB:")
try:
    rows = session.execute(query)
    for row in rows:
        print(row)
except Exception as e:
    print("error while reading:", e)

data insight CassandraDB:
Row(event_date=Date(20062), campaign_name='1001', ctr=0.048192769289016724, total_clicks=8, total_impressions=166)
Row(event_date=Date(20062), campaign_name='1006', ctr=0.06334841996431351, total_clicks=14, total_impressions=221)
Row(event_date=Date(20062), campaign_name='1007', ctr=0.05076142027974129, total_clicks=10, total_impressions=197)
Row(event_date=Date(20062), campaign_name='1009', ctr=0.032608695328235626, total_clicks=6, total_impressions=184)
Row(event_date=Date(20062), campaign_name='1010', ctr=0.05232558026909828, total_clicks=9, total_impressions=172)
Row(event_date=Date(20062), campaign_name='102', ctr=0.05213269963860512, total_clicks=11, total_impressions=211)
Row(event_date=Date(20062), campaign_name='103', ctr=0.05747126415371895, total_clicks=15, total_impressions=261)
Row(event_date=Date(20062), campaign_name='105', ctr=0.055084746330976486, total_clicks=13, total_impressions=236)
Row(event_date=Date(20062), campaign_name='114', ctr=0.04

In [18]:
print("Executing Task 2: Top 5 advertisers by spend...")

# Group by AdvertiserName and sum the AdCost
task2_df = df.groupby('AdvertiserName')['AdCost'].sum().reset_index()
top5_adv = task2_df.sort_values(by='AdCost', ascending=False).head(5)

# Insert the top 5 into Cassandra
for _, row in top5_adv.iterrows():
    session.execute(insert_q2, (TIME_WINDOW, float(row['AdCost']), str(row['AdvertiserName'])))

print("Task 2 completed!")

Executing Task 2: Top 5 advertisers by spend...
Task 2 completed!


In [19]:
#check the results for 2nd task
query = "SELECT * FROM top_advertisers_by_spend LIMIT 10;"

print("data insight CassandraDB:")
try:
    rows = session.execute(query)
    for row in rows:
        print(row)
except Exception as e:
    print("error while reading:", e)

data insight CassandraDB:
Row(time_window='last_30_days', total_spend=338012.03125, advertiser_name='Advertiser_11')
Row(time_window='last_30_days', total_spend=337651.34375, advertiser_name='Advertiser_91')
Row(time_window='last_30_days', total_spend=336557.25, advertiser_name='Advertiser_99')
Row(time_window='last_30_days', total_spend=336507.84375, advertiser_name='Advertiser_45')
Row(time_window='last_30_days', total_spend=335649.0625, advertiser_name='Advertiser_52')


In [13]:
# I took 1m rows for faster downloading- hope it's not a problem
from cassandra.query import BatchStatement, BatchType
import numpy as np

print("Executing Task 3: Optimized Batch Insertion (Correct Attributes)...")

# 1. Limit and prepare data
df_limited = df.head(1000000).copy()
df_limited['was_clicked_bool'] = df_limited['WasClicked'].apply(lambda x: True if x in [1, '1', True, 'True'] else False)

# 2. Extract records using
records = df_limited[['UserID', 'event_timestamp', 'CampaignName', 'was_clicked_bool']].to_dict('records')

# 3. Batch processing settings
BATCH_SIZE = 50
total_records = len(records)

print(f"Starting batch insertion... Total batches: {total_records // BATCH_SIZE}")

for i in range(0, total_records, BATCH_SIZE):
    # Use UNLOGGED batch for maximum speed
    batch = BatchStatement(batch_type=BatchType.UNLOGGED)

    chunk = records[i : i + BATCH_SIZE]
    for row in chunk:
        batch.add(insert_q3, (
            int(row['UserID']),             # Changed to UserID
            row['event_timestamp'],
            str(row['CampaignName']),
            bool(row['was_clicked_bool'])
        ))

    # Send the batch to Cassandra
    session.execute(batch)

    if i % 50000 == 0:
        print(f"Progress: {i}/{total_records} rows processed...")

print(f"Task 3 completed! Successfully processed 1M rows.")

Executing Task 3: Optimized Batch Insertion (Correct Attributes)...
Starting batch insertion... Total batches: 20000
Progress: 0/1000000 rows processed...
Progress: 50000/1000000 rows processed...
Progress: 100000/1000000 rows processed...
Progress: 150000/1000000 rows processed...
Progress: 200000/1000000 rows processed...
Progress: 250000/1000000 rows processed...
Progress: 300000/1000000 rows processed...
Progress: 350000/1000000 rows processed...
Progress: 400000/1000000 rows processed...
Progress: 450000/1000000 rows processed...
Progress: 500000/1000000 rows processed...
Progress: 550000/1000000 rows processed...
Progress: 600000/1000000 rows processed...
Progress: 650000/1000000 rows processed...
Progress: 700000/1000000 rows processed...
Progress: 750000/1000000 rows processed...
Progress: 800000/1000000 rows processed...
Progress: 850000/1000000 rows processed...
Progress: 900000/1000000 rows processed...
Progress: 950000/1000000 rows processed...
Task 3 completed! Successfull

In [15]:
#check the results for 3nd task

query = "SELECT * FROM user_ad_history WHERE user_id = 302602 LIMIT 10;"

print("data insight CassandraDB:")
try:
    rows = session.execute(query)
    for row in rows:
        print(row)
except Exception as e:
    print("error while reading:", e)

data insight CassandraDB:
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 11, 23, 9, 33, 40), campaign_name='Campaign_291', was_clicked=False)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 11, 16, 4, 50, 12), campaign_name='Campaign_681', was_clicked=False)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 11, 9, 19, 3, 14), campaign_name='Campaign_348', was_clicked=False)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 11, 2, 5, 42, 56), campaign_name='Campaign_808', was_clicked=True)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 10, 22, 10, 50, 45), campaign_name='Campaign_468', was_clicked=False)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 10, 7, 17, 50, 55), campaign_name='Campaign_518', was_clicked=True)
Row(user_id=302602, event_timestamp=datetime.datetime(2024, 10, 5, 8, 34, 55), campaign_name='Campaign_297', was_clicked=False)


In [16]:
print("Executing Task 4: Top 10 users by clicks...")

# Filtering and Aggregation
clicks_only = df[df['WasClicked'].isin([1, '1', True, 'True'])]

# Group by UserID and count clicks
task4_df = clicks_only.groupby('UserID').size().reset_index(name='total_clicks')
top10_users = task4_df.sort_values(by='total_clicks', ascending=False).head(10)

print(f"Step 1: Top 10 users identified. Preparation for upload starting...")

# Prepare data for Batch
records = top10_users.to_dict('records')
total_records = len(records)

# Batch Execution
batch = BatchStatement(batch_type=BatchType.UNLOGGED)

for row in records:
    batch.add(insert_q4, (
        TIME_WINDOW,
        int(row['total_clicks']),
        int(row['UserID'])
    ))

# Send to Cassandra
try:
    session.execute(batch)
    print(f"Step 2: Batch execution successful. Processed {total_records} records.")
except Exception as e:
    print(f"Error during batch execution: {e}")

print("Task 4 completed!")

Executing Task 4: Top 10 users by clicks...
Step 1: Top 10 users identified. Preparation for upload starting...
Step 2: Batch execution successful. Processed 10 records.
Task 4 completed!


In [17]:
#check the results for 4th task

query = "SELECT * FROM top_users_by_clicks LIMIT 10;"

print("data insight CassandraDB:")
try:
    rows = session.execute(query)
    for row in rows:
        print(row)
except Exception as e:
    print("error while reading:", e)

data insight CassandraDB:
Row(time_window='last_30_days', total_clicks=7, user_id=470)
Row(time_window='last_30_days', total_clicks=7, user_id=59472)
Row(time_window='last_30_days', total_clicks=7, user_id=101187)
Row(time_window='last_30_days', total_clicks=7, user_id=218358)
Row(time_window='last_30_days', total_clicks=7, user_id=617361)
Row(time_window='last_30_days', total_clicks=6, user_id=405681)
Row(time_window='last_30_days', total_clicks=6, user_id=415324)
Row(time_window='last_30_days', total_clicks=6, user_id=419772)
Row(time_window='last_30_days', total_clicks=6, user_id=529323)
Row(time_window='last_30_days', total_clicks=6, user_id=574570)


In [18]:
print("Executing Task 5: Top 5 advertisers by region...")
# Group by region and AdvertiserName, sum AdCost

task5_df = df.groupby(['region', 'AdvertiserName'])['AdCost'].sum().reset_index()
# Sort and pick top 5 for each region based on AdCost

top5_per_region = task5_df.sort_values(['region', 'AdCost'], ascending=[True, False]).groupby('region').head(5)
# Insert into Cassandra

for _, row in top5_per_region.iterrows():
    session.execute(insert_q5, (
        str(row['region']),
        TIME_WINDOW,
        float(row['AdCost']),
        str(row['AdvertiserName'])
    ))
print("Task 5 completed!")



Executing Task 5: Top 5 advertisers by region...
Task 5 completed!


In [23]:
#check the results for 5th task

query = "SELECT * FROM top_advertisers_by_region LIMIT 10;"

print("data insight CassandraDB:")
try:
    rows = session.execute(query)
    for row in rows:
        print(row)
except Exception as e:
    print("error while reading:", e)

data insight CassandraDB:
Row(region='USA', time_window='last_30_days', total_spend=68405.21875, advertiser_name='Advertiser_11')
Row(region='USA', time_window='last_30_days', total_spend=67732.078125, advertiser_name='Advertiser_52')
Row(region='USA', time_window='last_30_days', total_spend=67482.15625, advertiser_name='Advertiser_19')
Row(region='USA', time_window='last_30_days', total_spend=67354.21875, advertiser_name='Advertiser_45')
Row(region='USA', time_window='last_30_days', total_spend=67084.859375, advertiser_name='Advertiser_99')
Row(region='Australia', time_window='last_30_days', total_spend=68023.15625, advertiser_name='Advertiser_91')
Row(region='Australia', time_window='last_30_days', total_spend=67456.953125, advertiser_name='Advertiser_11')
Row(region='Australia', time_window='last_30_days', total_spend=67440.921875, advertiser_name='Advertiser_66')
Row(region='Australia', time_window='last_30_days', total_spend=67403.953125, advertiser_name='Advertiser_99')
Row(regio